In [1]:
# !pip install transformers datasets -q

## This uses roberta-base as an example, but any model can be used. When changing models, replace all instances of Roberta with the new model name (Electra, Bert, Albert, etc.). If you run into errors, refer to the source code for that specific model (XXXForSequenceClassification) and compare it.

For example, if you use Deberta, check how the `__init__` method ([link](https://github.com/huggingface/transformers/blob/93d3fd86459eb27bc584da29a3d542817a395bca/src/transformers/models/deberta/modeling_deberta.py#L1131)) and `forward` method ([link](https://github.com/huggingface/transformers/blob/93d3fd86459eb27bc584da29a3d542817a395bca/src/transformers/models/deberta/modeling_deberta.py#L1162)) are different. There might be 1 or 2 parameters that need to be changed to have it work properly.


In [1]:
import torch
from torch import nn
from transformers import ModernBertModel, ModernBertForSequenceClassification
from transformers.modeling_outputs import SequenceClassifierOutput
from typing import Optional, Union, Tuple
from transformers import set_seed

set_seed(42)

class ClassificationHead(nn.Module):
    """Head for sentence-level classification tasks."""

    def __init__(self, config, num_extra_dims):
        super().__init__()
        total_dims = config.hidden_size + num_extra_dims
        self.dense = nn.Linear(total_dims, total_dims)
        classifier_dropout = (
            config.classifier_dropout if config.classifier_dropout is not None else config.hidden_dropout_prob
        )
        self.dropout = nn.Dropout(classifier_dropout)
        self.out_proj = nn.Linear(total_dims, config.num_labels)

    def forward(self, features, **kwargs):
        x = self.dropout(features)
        x = self.dense(x)
        x = torch.tanh(x)
        x = self.dropout(x)
        x = self.out_proj(x)
        return x

class CustomSequenceClassification(ModernBertForSequenceClassification):

    def __init__(self, config, num_extra_dims):
        super().__init__(config)
        self.num_labels = config.num_labels
        self.config = config

        # ModernBERT uses 'model' as the attribute name
        self.model = ModernBertModel(config)
        self.classifier = ClassificationHead(config, num_extra_dims)

        # Initialize weights and apply final processing
        self.post_init()

    def forward(
        self,
        input_ids: Optional[torch.LongTensor] = None,
        attention_mask: Optional[torch.FloatTensor] = None,
        cefr_vec: Optional[torch.FloatTensor] = None,
        token_type_ids: Optional[torch.LongTensor] = None,
        position_ids: Optional[torch.LongTensor] = None,
        head_mask: Optional[torch.FloatTensor] = None,
        inputs_embeds: Optional[torch.FloatTensor] = None,
        labels: Optional[torch.LongTensor] = None,
        output_attentions: Optional[bool] = None,
        output_hidden_states: Optional[bool] = None,
        return_dict: Optional[bool] = None,
    ) -> Union[Tuple, SequenceClassifierOutput]:
        return_dict = return_dict if return_dict is not None else self.config.use_return_dict

        outputs = self.model(
            input_ids,
            attention_mask=attention_mask,
            position_ids=position_ids,
            inputs_embeds=inputs_embeds,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict,
        )

        sequence_output = outputs[0]
        cls_embedding = sequence_output[:, 0, :]
        output = torch.cat((cls_embedding, cefr_vec), dim=-1)
        logits = self.classifier(output)

        loss = None
        if labels is not None:
            if self.config.problem_type is None:
                if self.num_labels == 1:
                    self.config.problem_type = "regression"
                elif self.num_labels > 1 and (labels.dtype == torch.long or labels.dtype == torch.int):
                    self.config.problem_type = "single_label_classification"
                else:
                    self.config.problem_type = "multi_label_classification"

            if self.config.problem_type == "regression":
                loss_fct = nn.MSELoss()
                if self.num_labels == 1:
                    loss = loss_fct(logits.squeeze(), labels.squeeze())
                else:
                    loss = loss_fct(logits, labels)
            elif self.config.problem_type == "single_label_classification":
                loss_fct = nn.CrossEntropyLoss()
                loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))
            elif self.config.problem_type == "multi_label_classification":
                loss_fct = nn.BCEWithLogitsLoss()
                loss = loss_fct(logits, labels)

        if not return_dict:
            output = (logits,) + outputs[2:]
            return ((loss,) + output) if loss is not None else output

        return SequenceClassifierOutput(
            loss=loss,
            logits=logits,
            hidden_states=outputs.hidden_states,
            attentions=outputs.attentions,
        )

### This example does 3-class classification and adds 5 extra data points for each example. Please change for your own use case.



For simple text classification, the last hidden state for the CLS token is run through a classifier, producing a score for each label.  A simple way to combine text and numerical/categorical data is to concatenate the CLS embedding with the extra data. If the CLS embedding is [1.0, 2.0, 3.0] and the extra data is 5.0, then the concatenated version is [1.0, 2.0, 3.0, 5.0]. Likewise for categorical data, the variable can be turned into a one-hot encoding and concatenated. The concatenated version is then put through a classifier to produce scores for each label.

In [2]:
from transformers import AutoConfig

set_seed(42)

# Initialize new model
config = AutoConfig.from_pretrained("answerdotai/ModernBERT-base", num_labels=4)
new_model = CustomSequenceClassification(config, num_extra_dims=6)

# Load pretrained weights into the base model only
pretrained = ModernBertModel.from_pretrained("answerdotai/ModernBERT-base")
new_model.model.load_state_dict(pretrained.state_dict())

<All keys matched successfully>

## This is just dummy data, but you should make a dataset where each example has text, the extra features, and a label

After the text gets tokenized and run through the model, the output embeddings will be concatenated with the extra data. That concatenation will then go through the classifer and produce scores for each label.


NOTE: Your `extra_data` should be one hot data or scaled to mean 0, variance 1 if possible.

In [3]:
from datasets import Dataset, load_dataset

ds = load_dataset("json", data_files=r"C:\Users\coope\Documents\Learner_Writing_Classification\cefr_lexical_sophistication\cefr-lexical-sophistication-main\wai_data\train_cefr_AW.jsonl", split="train")
# convert CEFR vocab data to a vector and add as cefr_vec
cefr_keys = ["cefr_A1", "cefr_A2", "cefr_B1", "cefr_B2", "cefr_C1", "cefr_C2"]
ds = ds.map(lambda x: {"cefr_vec": [x[k] for k in cefr_keys]})
ds = ds.remove_columns(cefr_keys)

# Keep only the columns you need
ds = ds.select_columns(["text", "label", "cefr_vec"])

# convert CEFR levels to integers in a new "labels" column
label_map = {"A2": 0, "B1": 1, "B2": 2, "C1": 3}
ds = ds.map(lambda x: {"labels": label_map[x["label"]]})

# remove the old "label" column
ds = ds.remove_columns(["label"])

print(ds)
ds[0]

Dataset({
    features: ['text', 'cefr_vec', 'labels'],
    num_rows: 4195
})


{'text': "New Year is always an invite to establish goals for the next 365 days. There are some areas there I'm pleased like improving my idioms knowledge, writing, speaking, presenting scientific articles. There are also areas that I should improve like statistics, IA, algorithms and how to present a work I developed. So I need to have some strategies for achieving goals. 1 - Improving a new language: I need to study, daily, at least little time like 15 minutes. And this study includes writing, reading a lot. 2 - Presenting scientific articles: It's important to reach innovative findings, read, study and try to disseminate these new ideas. When I consider to be a presenter I need to study a lot. 3 - Improve my statistics knowledge: a challenge, but not impossible. As to achieve previous goals, I'll need to study sistematically and never give up. 4 - Understanding about algorithms - it's a new knowledge and very difficult to study for whom is not a specialty on this area but I think th

In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")

tokenized_ds = ds.map(lambda x: tokenizer(x["text"]))

In [5]:
# remove "text" column, as text has been tokenized and it is no longer needed
tokenized_ds = tokenized_ds.remove_columns(["text"])

# here is what one example looks like
tokenized_ds[0]

{'cefr_vec': [0.4293478261,
  0.2554347826,
  0.1793478261,
  0.0815217391,
  0.0054347826,
  0.0489130435],
 'labels': 1,
 'input_ids': [50281,
  4257,
  10519,
  310,
  1900,
  271,
  19864,
  281,
  5100,
  7342,
  323,
  253,
  1735,
  23412,
  1897,
  15,
  1707,
  403,
  690,
  3672,
  627,
  309,
  1353,
  13864,
  751,
  11138,
  619,
  22467,
  3056,
  3640,
  13,
  4028,
  13,
  8288,
  13,
  15250,
  8249,
  7774,
  15,
  1707,
  403,
  671,
  3672,
  326,
  309,
  943,
  3157,
  751,
  9990,
  13,
  32815,
  13,
  11333,
  285,
  849,
  281,
  1246,
  247,
  789,
  309,
  3715,
  15,
  1893,
  309,
  878,
  281,
  452,
  690,
  8130,
  323,
  17170,
  7342,
  15,
  337,
  428,
  3173,
  40037,
  247,
  747,
  3448,
  27,
  309,
  878,
  281,
  1263,
  13,
  5312,
  13,
  387,
  1878,
  1652,
  673,
  751,
  1458,
  2909,
  15,
  1244,
  436,
  1263,
  3797,
  4028,
  13,
  4361,
  247,
  2257,
  15,
  374,
  428,
  23952,
  272,
  8249,
  7774,
  27,
  733,
  434,
  1774,
 

## If you set it up this way, you can train it using the Hugging Face Trainer, and the process is nearly identical to the example script here: https://github.com/huggingface/transformers/blob/master/examples/pytorch/text-classification/run_glue.py

In [6]:
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding
import torch

class CustomDataCollator(DataCollatorWithPadding):
    def __call__(self, features):
        # Extract cefr_vec before the parent collator processes everything
        cefr_vec = [f.pop("cefr_vec") for f in features]
        
        # Let the parent handle input_ids, attention_mask, labels
        batch = super().__call__(features)
        
        # Add cefr_vec back as a tensor
        batch["cefr_vec"] = torch.tensor(cefr_vec, dtype=torch.float)
        
        return batch

data_collator = CustomDataCollator(tokenizer=tokenizer)

# the best parameters are included here after trying multiple options   
args = TrainingArguments(
    save_strategy="no",
    output_dir="./modernbert-AW",
    seed=42,
    learning_rate=1e-05,
    num_train_epochs=3,
    remove_unused_columns=False,
)

trainer = Trainer(
    model=new_model, 
    train_dataset=tokenized_ds, 
    data_collator=data_collator,
    args=args,
)

In [7]:
trainer.train()

Step,Training Loss
500,0.881200
1000,0.677400
1500,0.527000


TrainOutput(global_step=1575, training_loss=0.6864412822420635, metrics={'train_runtime': 131.4393, 'train_samples_per_second': 95.748, 'train_steps_per_second': 11.983, 'total_flos': 2623844669215560.0, 'train_loss': 0.6864412822420635, 'epoch': 3.0})

In [8]:
from datasets import load_dataset
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix
import plotly.graph_objects as go
import numpy as np
import torch

# Load and preprocess validation data
val_ds = load_dataset("json", data_files=r"C:\Users\coope\Documents\Learner_Writing_Classification\cefr_lexical_sophistication\cefr-lexical-sophistication-main\wai_data\val_cefr_AW.jsonl", split="train")
# convert CEFR vocab data to a vector and add as cefr_vec
cefr_keys = ["cefr_A1", "cefr_A2", "cefr_B1", "cefr_B2", "cefr_C1", "cefr_C2"]
val_ds = val_ds.map(lambda x: {"cefr_vec": [x[k] for k in cefr_keys]})
val_ds = val_ds.remove_columns(cefr_keys)

val_ds = val_ds.select_columns(["text", "label", "cefr_vec"])

label_map = {"A2": 0, "B1": 1, "B2": 2, "C1": 3}
id_to_label = {0: "A2", 1: "B1", 2: "B2", 3: "C1"}

val_ds = val_ds.map(lambda x: {"labels": label_map[x["label"]]})
val_ds = val_ds.remove_columns(["label"])

# Tokenize
tokenized_val_ds = val_ds.map(lambda x: tokenizer(x["text"]))
tokenized_val_ds = tokenized_val_ds.remove_columns(["text"])

# Get predictions
new_model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
new_model.to(device)

all_preds = []
all_labels = []

with torch.no_grad():
    for example in tokenized_val_ds:
        input_ids = torch.tensor([example["input_ids"]]).to(device)
        attention_mask = torch.tensor([example["attention_mask"]]).to(device)
        cefr_vec = torch.tensor([example["cefr_vec"]], dtype=torch.float).to(device)
        
        outputs = new_model(input_ids=input_ids, attention_mask=attention_mask, cefr_vec=cefr_vec)
        pred = torch.argmax(outputs.logits, dim=-1).item()
        
        all_preds.append(pred)
        all_labels.append(example["labels"])

# Calculate metrics
precision, recall, f1, support = precision_recall_fscore_support(all_labels, all_preds, average=None)
labels = ["A2", "B1", "B2", "C1"]

print("Per-class metrics:")
for i, label in enumerate(labels):
    print(f"{label}: Precision={precision[i]:.3f}, Recall={recall[i]:.3f}, F1={f1[i]:.3f}")

# Overall metrics
precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(all_labels, all_preds, average="macro")
print(f"\nMacro average: Precision={precision_macro:.3f}, Recall={recall_macro:.3f}, F1={f1_macro:.3f}")

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
# Normalize by row (true labels) to get proportions
cm_normalized = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]

# Create text annotations with both count and percentage
annotations = []
for i in range(len(labels)):
    row = []
    for j in range(len(labels)):
        row.append(f"{cm_normalized[i][j]*100:.1f}%<br>({cm[i][j]})")
    annotations.append(row)

# Create plotly heatmap (black = high, white = low)
fig = go.Figure(data=go.Heatmap(
    z=cm_normalized,
    x=labels,
    y=labels,
    colorscale=[[0, "white"], [1, "black"]],
    text=annotations,
    texttemplate="%{text}",
    textfont={"size": 12},
    hovertemplate="True: %{y}<br>Predicted: %{x}<br>Proportion: %{z:.2f}<extra></extra>"
))

fig.update_layout(
    title="ModernBERT + CEFR Vocabulary (AW)",
    xaxis_title="Predicted",
    yaxis_title="True",
    xaxis={"side": "bottom"},
    yaxis={"autorange": "reversed"},
    width=500,
    height=500
)

fig.show()

Per-class metrics:
A2: Precision=0.846, Recall=0.767, F1=0.805
B1: Precision=0.716, Recall=0.643, F1=0.678
B2: Precision=0.539, Recall=0.739, F1=0.624
C1: Precision=0.700, Recall=0.447, F1=0.545

Macro average: Precision=0.700, Recall=0.649, F1=0.663


# Save Model

In [9]:
new_model.save_pretrained(r"C:\Users\coope\Documents\Learner_Writing_Classification\cefr_lexical_sophistication\cefr-lexical-sophistication-main\models\ModernBERT_AW")
tokenizer.save_pretrained(r"C:\Users\coope\Documents\Learner_Writing_Classification\cefr_lexical_sophistication\cefr-lexical-sophistication-main\models\ModernBERT_AW")

('C:\\Users\\coope\\Documents\\Learner_Writing_Classification\\cefr_lexical_sophistication\\cefr-lexical-sophistication-main\\models\\ModernBERT_AW\\tokenizer_config.json',
 'C:\\Users\\coope\\Documents\\Learner_Writing_Classification\\cefr_lexical_sophistication\\cefr-lexical-sophistication-main\\models\\ModernBERT_AW\\special_tokens_map.json',
 'C:\\Users\\coope\\Documents\\Learner_Writing_Classification\\cefr_lexical_sophistication\\cefr-lexical-sophistication-main\\models\\ModernBERT_AW\\tokenizer.json')

# Load Model

In [10]:
from transformers import AutoTokenizer
from safetensors.torch import load_file

tokenizer = AutoTokenizer.from_pretrained(r"C:\Users\coope\Documents\Learner_Writing_Classification\cefr_lexical_sophistication\cefr-lexical-sophistication-main\models\ModernBERT_AW")

# For your custom model, you'll need to load it manually since it has extra args
config = AutoConfig.from_pretrained(r"C:\Users\coope\Documents\Learner_Writing_Classification\cefr_lexical_sophistication\cefr-lexical-sophistication-main\models\ModernBERT_AW")
loaded_model = CustomSequenceClassification(config, num_extra_dims=6)
state_dict = load_file(r"C:\Users\coope\Documents\Learner_Writing_Classification\cefr_lexical_sophistication\cefr-lexical-sophistication-main\models\ModernBERT_AW\model.safetensors")
loaded_model.load_state_dict(state_dict)

<All keys matched successfully>